# Prostate158 RQ2 — finalize: zone volumes + corrected figures (Colab)

Two jobs that both need the Drive data and nothing else. **No GPU, no model,
no inference** — this notebook never loads the checkpoint, so it cannot change
any segmentation result.

1. **Zone volumes against the annotation** — finishes task C. Adds per-case
   error, bias and Bland–Altman limits of agreement to the predicted volumes
   already computed locally.
2. **Redraw the qualitative figures** — the saved 3D projections were rendered
   with square voxels, which compresses the coronal and sagittal panels by the
   spacing ratio (~6.4×) and makes a geometrically valid reconstruction look
   like a flat disc. The renderer is fixed; these figures predate the fix.

A CPU runtime is enough and is faster to get. Runtime → Change runtime type →
CPU, then Run all.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration (edit here)

In [ ]:
REPO_URL     = "https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git"
REPO_BRANCH  = "main"
PROJECT_DIR  = "/content/prostate-mri-segmentation-thesis"

DRIVE_BASE   = "/content/drive/MyDrive/THESIS_PROSTATE158"

# The completed RQ2 run (E1, epoch 88). Read, never overwritten except for the
# figures under visualizations/ and the volume outputs below.
RQ2_DIR      = f"{DRIVE_BASE}/results/rq2_e1_epoch88"

# Where the official 19-case test archive was extracted. Section 5 descends
# automatically to the folder that directly holds the case directories.
TEST_DIR     = f"{DRIVE_BASE}/dataset/test/extracted"

VOLUMES_OUT  = f"{DRIVE_BASE}/results/rq2_zone_volumes"
MASK_NAME    = "t2_anatomy_reader1.nii.gz"
EXPECTED_TEST_CASES = 19

for k in ["PROJECT_DIR", "DRIVE_BASE", "RQ2_DIR", "TEST_DIR", "VOLUMES_OUT"]:
    print(f"{k:12s} = {globals()[k]}")

## 3. Repository setup (clone or pull — no duplicates)

In [ ]:
import os, subprocess

def sh(*args, check=True):
    print("$", " ".join(args))
    return subprocess.run(args, check=check)

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_DIR)
else:
    sh("git", "-C", PROJECT_DIR, "fetch", "origin", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "checkout", REPO_BRANCH)
    sh("git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", REPO_BRANCH)

os.chdir(PROJECT_DIR)
print("\ncwd:", os.getcwd())
sh("git", "-C", PROJECT_DIR, "log", "--oneline", "-1")

## 4. Install dependencies

No torch needed — nothing here runs the model.

In [ ]:
%pip install -q nibabel>=5.4 scipy>=1.13 pyyaml>=6.0 pandas>=2.0
import nibabel, numpy, scipy, pandas, matplotlib
print("nibabel", nibabel.__version__, "| numpy", numpy.__version__, "| scipy", scipy.__version__)

## 5. Verify paths and resolve the test case-root

In [ ]:
import os, sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "scripts"))

problems = []
if not os.path.isdir(DRIVE_BASE):
    problems.append(f"Drive base not found: {DRIVE_BASE} (is Drive mounted?)")
if not os.path.isdir(os.path.join(RQ2_DIR, "reconstructions_3d")):
    problems.append(f"No reconstructions_3d/ under {RQ2_DIR} — is the RQ2 run on Drive?")
if not os.path.isdir(TEST_DIR):
    problems.append(f"TEST_DIR not found: {TEST_DIR}")
if problems:
    raise FileNotFoundError("Environment check failed:" + "".join(
        chr(10) + "  - " + p for p in problems))

from audit_test_archive import find_case_root
TEST_CASE_ROOT = find_case_root(TEST_DIR)

n_recon = len([f for f in os.listdir(os.path.join(RQ2_DIR, "reconstructions_3d"))
               if f.endswith(".nii.gz")])
case_dirs = sorted(d for d in os.listdir(TEST_CASE_ROOT)
                   if os.path.isdir(os.path.join(TEST_CASE_ROOT, d)))

print(f"TEST_CASE_ROOT  = {TEST_CASE_ROOT}")
print(f"test cases      = {len(case_dirs)}  (expected {EXPECTED_TEST_CASES})")
print(f"reconstructions = {n_recon}  (expected {EXPECTED_TEST_CASES})")

assert len(case_dirs) == EXPECTED_TEST_CASES, (
    f"Resolved case-root holds {len(case_dirs)} folders, expected {EXPECTED_TEST_CASES}.")
assert n_recon == EXPECTED_TEST_CASES, (
    f"{RQ2_DIR}/reconstructions_3d holds {n_recon} volumes, expected {EXPECTED_TEST_CASES}. "
    "A partial Drive sync is the likely cause.")
print("\nOK — cohort complete.")

## 6. Zone volumes against the annotation (finishes task C)

Volume is voxel count × |det(affine)|, so these numbers are only meaningful
because the reconstruction restored the source affine exactly. Bias is
mean(prediction − ground truth); the **limits of agreement**, not the bias, are
what decide whether a volume is usable for an individual patient.

In [ ]:
# Colab expands {NAME} from the Python namespace inside a ! line.
!python scripts/run_zone_volumes.py \
    --recon-dir  "{RQ2_DIR}/reconstructions_3d" \
    --gt-root    "{TEST_CASE_ROOT}" \
    --mask-name  "{MASK_NAME}" \
    --output-dir "{VOLUMES_OUT}"

## 7. Redraw the qualitative figures with the corrected aspect ratio

In [ ]:
!python scripts/regenerate_rq2_figures.py \
    --rq2-dir   "{RQ2_DIR}" \
    --gt-root   "{TEST_CASE_ROOT}" \
    --mask-name "{MASK_NAME}"

## 8. Look at the results

In [ ]:
import os
from IPython.display import Markdown, Image, display

report = os.path.join(VOLUMES_OUT, "report.md")
if os.path.exists(report):
    display(Markdown(open(report, encoding="utf-8").read()))

viz = os.path.join(RQ2_DIR, "visualizations")
for pid in sorted(os.listdir(viz)):
    proj = os.path.join(viz, pid, f"{pid.replace('patient_', 'patient_')}_3d_projections.png")
    if os.path.exists(proj):
        print(pid)
        display(Image(filename=proj, width=950))

## 9. What changed, and what did not

- The **segmentation metrics are untouched.** This notebook never loaded the
  model. `metrics_per_case.csv`, `metrics_summary.csv`, `summary.json` and the
  reconstructed volumes are exactly as the RQ2 run left them.
- The figures under `visualizations/` were **overwritten** with correctly
  proportioned versions. The coronal and sagittal panels should now show a
  walnut, not a pancake.
- `results/rq2_zone_volumes/` now carries agreement figures. Until this ran,
  those volumes were model output with no accuracy claim attached.